# N150 · 快照、历史查询与可持久化

**目标：** 在保存历史版本时复用未变部分且不污染旧结果。

**先修：** N042, N005。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** 按索引版本历史；二分；稀疏记录；结构共享；路径复制；持久化线段树拓展。

**配套讲解来源：** [同名逐章意见](../../comment/150_snapshots_persistent_versions.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

想象你在编辑一份文档，想随时"拍快照"存档，之后还能查看"在第 k 次存档时，某个位置写的是什么"。最直观的做法是每次拍快照就把整个数组完整复制一份。但这样做有个大问题：一个 100 万长度的数组哪怕只改了 1 个位置，拍一次快照也得复制 100 万个数，绝大多数复制完全是浪费——多数位置压根没变。

本章的问题就是：**怎么在保留所有历史版本的同时，不复制没变化的部分、也不让新的修改污染旧版本？** 接口三个操作：

- `set(index, val)`：在"当前工作版本"里把 index 位置改成 val（还没拍快照，随时能改）；
- `snap()`：拍一张快照，返回一个快照编号 `snap_id`（从 0 开始，每次加一）；
- `get(index, snap_id)`：查询"在编号为 snap_id 的那次快照里，index 位置是什么值"。

举个具体到数字的例子（就是 “运行示例”部分 的小实例）：建长度 3 的数组，`set(0,5)`，拍快照得 `first=0`；接着 `set(0,6)`、`set(0,7)`，再拍快照得 `second=1`。那么 `get(0, 0)` 应该是 **5**（第一次快照时 0 号位只写过 5），`get(0, 1)` 应该是 **7**（第二次快照前连写 6 和 7，同一段未提交时间里只保留最后的 7）。为什么不是 6？因为 6 和 7 都写在两次快照之间，快照 1 拍下时 6 已经被 7 覆盖了。

## 2. 基础知识：先读懂这些词

- **快照（snapshot）与版本号（snap_id）**：快照不是"复制一份数组"，而只是一个递增的整数编号。编号把时间切成一段一段：编号 `k` 的快照拍下后、编号 `k+1` 拍下前发生的所有 `set`，都属于"版本 k+1 的未提交修改"。
- **稀疏记录（sparse history）**：每个位置各自维护一个小账本，只记"这个位置被改过的地方"——`(版本号, 值)` 组成的列表。没被改过的位置账本里只有一条初始记录 `(0, 0)`（初始版本 0，值 0）。数组再大，没动过的位置也不多占一个字。
- **结构共享（structure sharing）**：新版本不复制旧数据，而是和旧版本共用没变的部分。本章里"共享"体现在：旧快照的记录永远原地不动，新修改只是往账本后面追加，所有旧版本自然同时有效。
- **时间前驱查询**：查"版本 snap_id 时这个位置的值"，等价于在账本里找"版本号不大于 snap_id 的最后一条记录"。因为一个位置两次修改之间值保持不变，最后一条不晚于 snap_id 的记录就是当时的值。
- **二分查找（`bisect_right`）**：账本按版本号从小到大排好序，所以"找最后一条版本 ≤ snap_id 的记录"可以用二分，不用从头扫。`bisect_right(...)-1` 一步定位。
- **路径复制（path copying）**：可持久化数据结构的通用招数——修改时只复制"从根到被改位置"这一条路径，其余子树与旧版本共享。本章不需要它，但 “正确性与复杂度”部分 提到它是持久化线段树支持"历史区间查询"的基础，属于拓展。
- **均摊 O(1)**：单次 `set` 偶尔要给列表追加元素（可能触发扩容），但一连串 `set` 平摊下来每次是常数代价。

## 3. 思路推导：从小例子开始

- **Step 1**：别在 `snap()` 时复制任何东西——`snap()` 只做"发一个新版本号"这一件事，O(1) 完成。真正要记录信息的是 `set`。
- **Step 2**：把账本开在每个位置上：`history[index]` 是这个位置的 `(版本, 值)` 列表。初始每个位置都是 `[(0, 0)]`（版本 0 值 0，覆盖"从未改过"的情形）。
- **Step 3**：`set(index, val)` 时看账本最后一条的版本号：如果它等于当前未提交的版本号，说明这一版已经写过这个位置，直接**原地覆盖**最后一条（同一快照前的多次 set 只留最终值）；否则追加一条 `(当前版本, val)`。
- **Step 4**：`get(index, snap_id)` 时在账本里二分，找到版本号 ≤ snap_id 的最后一条，返回它的值。
- **Step 5**：防呆检查：下标越界要报错；`snap_id` 必须是"已经拍过"的编号（小于当前版本计数），查未来快照直接拒绝。

**手算演示**（完整重演 “运行示例”部分 的例子，初始 `history = [[(0,0)], [(0,0)], [(0,0)]]`，当前版本 `current = 0`）：

1. `set(0, 5)`：0 号账本最后一条是 `(0,0)`，版本 0 == 当前版本 0 → 原地覆盖，账本变 `[(0,5)]`。
2. `snap()`：返回 0，`current` 变 1。所以 `first = 0`。
3. `set(0, 6)`：最后一条 `(0,5)` 版本 0 ≠ 当前版本 1 → 追加，账本变 `[(0,5), (1,6)]`。
4. `set(0, 7)`：最后一条 `(1,6)` 版本 1 == 当前版本 1 → 原地覆盖，账本变 `[(0,5), (1,7)]`。注意 6 消失了——它从未属于任何已提交快照，不留痕迹是对的。
5. `snap()`：返回 1，`current` 变 2。所以 `second = 1`。

把整个过程中"当前版本号 current"的变化也排出来，能看得更清楚——`set` 归属哪个版本，完全由执行那一刻 current 的值决定：

| 动作 | 执行前 current | 账本动作 | 执行后 0 号账本 |
|---|---|---|---|
| 初始 | 0 | 建账 `[(0,0)]` | `[(0,0)]` |
| `set(0,5)` | 0 | 末条版本相同 → 覆盖 | `[(0,5)]` |
| `snap()` 返回 0 | 0 → 1 | 只发号，不动账本 | `[(0,5)]` |
| `set(0,6)` | 1 | 末条版本不同 → 追加 | `[(0,5),(1,6)]` |
| `set(0,7)` | 1 | 末条版本相同 → 覆盖 | `[(0,5),(1,7)]` |
| `snap()` 返回 1 | 1 → 2 | 只发号，不动账本 | `[(0,5),(1,7)]` |

看这张表要抓两点：`snap` 两行账本纹丝不动（O(1) 的来源）；`set(0,6)` 和 `set(0,7)` 同处 current=1 的窗口，6 被 7 原地顶掉，从未真正属于任何已提交快照。

“运行示例”部分 的表格生成的正是这份状态：

| 索引 | 稀疏版本记录 | 快照0值 | 快照1值 |
|---|---|---|
| 0 | `[(0,5), (1,7)]` | 5 | 7 |
| 1 | `[(0,0)]` | 0 | 0 |
| 2 | `[(0,0)]` | 0 | 0 |

再看 `get` 怎么二分：查 `get(0, 0)` 时，在 `[(0,5), (1,7)]` 里找"版本 ≤ 0"的右边界，`bisect_right` 给出位置 1（插在 `(0,5)` 后面），减 1 得下标 0 → 值 5。查 `get(0, 1)` 时右边界是 2，减 1 得下标 1 → 值 7。1、2 号位置从未被写，任何快照查出来都是初始值 0。

**再补两个边界手算**：

- **查最新快照**：`current` 此时是 2，合法的 snap_id 是 0 和 1。`get(0, 1)` 用的是"已发放的最大编号"，二分命中 `(1,7)`——注意 1 号快照拍下之后我们**没有**再写过 0 号位，所以查到的 7 既是快照 1 的值也是"当前值"。
- **查还没拍的快照**：`get(0, 2)` 会撞上 `not 0<=2<self.current` 的检查（2 不小于 current=2），抛 `ValueError('snapshot must already be committed')`。快照编号必须先由 `snap()` 发放过才能查，"查未来"属于非法调用，实现选择当场报错而不是返回猜的值。

## 4. 核心代码：SnapshotArray

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
class SnapshotArray:

    def __init__(self, length):
        if length < 0:
            raise ValueError('nonnegative length required')
        self.history = [[(0, 0)] for _ in range(length)]
        self.current = 0

    def _check_index(self, index):
        if not 0 <= index < len(self.history):
            raise IndexError('index outside snapshot array')

    def set(self, index, val):
        self._check_index(index)
        history = self.history[index]
        if history[-1][0] == self.current:
            history[-1] = (self.current, val)
        else:
            history.append((self.current, val))

    def snap(self):
        result = self.current
        self.current += 1
        return result

    def get(self, index, snap_id):
        self._check_index(index)
        if not 0 <= snap_id < self.current:
            raise ValueError('snapshot must already be committed')
        history = self.history[index]
        position = bisect_right(history, snap_id, key=lambda item: item[0]) - 1
        return history[position][1]
```

### 逐段读懂代码

“分段实现”部分 的完整实现：

（对应程序见下方分段实现与完整代码。）

- `__init__`：`length<0` 时抛 `ValueError`（长度为 0 合法、为负不行）。`[[(0,0)] for _ in range(length)]` 是列表推导，给每个位置发一本只有一条初始记录的账本——**注意必须用推导式**，如果写 `[[(0,0)]]*length` 会让所有位置共享同一个列表对象，改一个位置会串到所有位置。`self.current` 是"下一个要发放的版本号"，从 0 开始。
- `_check_index`：下标防卫函数。`0<=index<len(...)` 这个链式比较等价于 `index>=0 and index<len(...)`，但更短。越界抛 `IndexError`，错误早暴露比悄悄吞掉好。
- `set`：先验下标，再取该位置的账本起个短名。核心分叉是 `if history[-1][0]==self.current:`——"账本最后一条的版本号是不是当前还没拍快照的这个版本"。是，就原地替换最后一条（`history[-1]=(self.current,val)` 用新元组换掉旧元组，因为元组本身不可改）；不是（说明当前版本这个位置还没写过），就追加一条。这个分叉同时保证了两件事：同一未提交版本内的多次写只留最后值；已提交版本（版本号更小的那些记录）永远不会被动到。
- `snap`：三连——记下当前版本号、版本号加一、返回记下的那个。第一次调用返回 0，第二次返回 1，以此类推。它完全不碰任何账本，这就是"snap O(1)"的底气。
- `get`：两道检查（下标合法、`snap_id` 落在 `[0, current)` 即必须是已发放的编号——查还没拍的快照抛 `ValueError`）。然后一行干完正事：`bisect_right(history, snap_id, key=lambda item: item[0])` 在账本里找"版本号 ≤ snap_id 的记录都排在其左侧"的分界位置，`-1` 之后 `position` 正是最后一条满足条件的记录下标，返回其值。`key=lambda item:item[0]` 告诉二分"比较时看元组第一项（版本号）"；这个 `key` 参数要求 Python 3.10 及以上，这是 “正确性与复杂度”部分 特别注明的前提。
- 整体数据结构上没有"撤销"或"修改历史"的接口，所有对过去的访问都是只读的——这就是"不污染旧结果"的结构保证。

## 5. 分段实现：按顺序运行

**本章接口：** `SnapshotArray.set/snap/get`

### 导入本章使用的库

In [1]:
from bisect import bisect_right

### SnapshotArray

In [2]:
class SnapshotArray:

    def __init__(self, length):
        if length < 0:
            raise ValueError('nonnegative length required')
        self.history = [[(0, 0)] for _ in range(length)]
        self.current = 0

    def _check_index(self, index):
        if not 0 <= index < len(self.history):
            raise IndexError('index outside snapshot array')

    def set(self, index, val):
        self._check_index(index)
        history = self.history[index]
        if history[-1][0] == self.current:
            history[-1] = (self.current, val)
        else:
            history.append((self.current, val))

    def snap(self):
        result = self.current
        self.current += 1
        return result

    def get(self, index, snap_id):
        self._check_index(index)
        if not 0 <= snap_id < self.current:
            raise ValueError('snapshot must already be committed')
        history = self.history[index]
        position = bisect_right(history, snap_id, key=lambda item: item[0]) - 1
        return history[position][1]

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [3]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

snapshots=SnapshotArray(3); snapshots.set(0,5); first=snapshots.snap(); snapshots.set(0,6); snapshots.set(0,7); second=snapshots.snap()
show_table(['索引','稀疏版本记录','快照0值','快照1值'],[(i,h,snapshots.get(i,first),snapshots.get(i,second)) for i,h in enumerate(snapshots.history)])

索引,稀疏版本记录,快照0值,快照1值
0,"[(0, 5), (1, 7)]",5,7
1,"[(0, 0)]",0,0
2,"[(0, 0)]",0,0


## 7. 正确性、适用条件与复杂度

一个位置在两次修改之间保持常值，因此历史查询等价于时间前驱查询。版本号单调，二分右界减一得到最后一个version≤snap_id的记录。同版本覆盖只影响尚未提交的状态；已提交版本记录不会被后续set覆盖。

**代价：** set均摊O(1)，snap O(1)，get O(log 该位置修改版本数)，空间O(长度+保留修改数)。持久化线段树可用路径复制支持历史区间查询，但属于拓展；本章只实现稀疏单点历史。Python要求支持bisect的key参数（3.10及以上）。

### 展开理解

**为什么查历史等价于"找最后一条不晚于 snap_id 的记录"？** 因为一个位置两次修改之间的值一直保持不变。账本按版本号从小到大记下了每一次生效的值，所以版本 `snap_id` 时刻的值，就是"最晚一次在 snap_id 之前（或恰好在 snap_id 那一版）生效的修改"的结果。举例：账本 `[(0,5),(1,7)]` 里查版本 0，最后一条不晚于 0 的记录是 `(0,5)`，所以是 5——7 是版本 1 才生效的，查版本 0 看不见它。

**为什么二分找得到？** 每条记录追加时的版本号只会更大（`current` 单调递增），同一版本又被原地合并成一条，所以账本里版本号严格递增、天然有序，二分的前提成立。`bisect_right` 给出"所有 ≤ snap_id 的记录"的右边界，减一正好是最后一条，不多不少。

**为什么旧版本不会被污染？** `set` 的原地覆盖只发生在"最后一条的版本号等于当前未提交版本"时。已提交版本的记录版本号一定小于当前版本（snap 之后 current 已经加过一），永远不会匹配这个条件，所以只会被追加、不会被改写。追加也不影响旧查询：旧查询的二分结果落在此前那些记录上，新追加的记录版本号更大，根本不参与。

**复杂度是多少？** 配具体数字说：

- `snap()`：三步赋值，O(1)。就算数组长 100 万，拍一万次快照也只是发一万个编号，一个元素都不复制（对比"整表复制"方案：100 万长度 × 1 万次 = 100 亿次数值拷贝，完全不可行）。
- `set()`：查下标、看账本最后一条、覆盖或追加，均摊 O(1)。
- `get()`：二分账本长度。账本最长也就是"这个位置被写过的版本数"，比如某位置总共被写过 20 次，`get` 就是 log 20 ≈ 5 步以内。一般写成 O(log 该位置修改版本数)。
- 空间：每个位置一本初始账本（长度 × 1 条）加上所有保留下来的修改记录，总共 O(数组长度 + 保留修改数)。100 万长度只改 1000 个位置，空间就是 100 万条初始记录 + 1000 条修改，而不是每次快照 100 万。

**能力边界（“正确性与复杂度”部分 特意声明）**：这套结构只支持"单点的历史查询"；要高效查"某个历史版本上某段区间的和"，得上持久化线段树（用路径复制让每个版本共享未改的子树），那是本章的拓展方向，不在实现范围内。

**对比一下三种存法的账单**，用"长度 1000 的数组、拍 1000 次快照、总共只改过 2000 次"来算：

| 方案 | 每次快照的代价 | 总空间 | 历史查询 |
|---|---|---|---|
| 整表复制 | O(1000) | 100 万个数 | O(1) |
| 每版本记差异字典 | O(1) 发号 + 记差异 | 取决于实现 | 遍历版本链，慢 |
| 本章稀疏账本 | O(1) 只发号 | 1000 条初始 + 2000 条修改 | 二分 O(log 20) 级别 |

稀疏账本赢在"改动多少存多少"：没改过的位置不花钱，改过多次但同一版本内的合并成一条，快照本身零成本。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的断言分三段：

- **第一段（三行小情景）**：
  - `array=SnapshotArray(3); assert array.snap()==0 and array.get(1,0)==0`：第一条测"首次快照编号是 0"（正常值），第二条测"从未写过的位置在任何快照里都是初始值 0"（默认值边界）。
  - `array.set(0,5); array.set(0,6); sid=array.snap(); assert array.get(0,sid)==6 and array.get(0,0)==0`：同一段未提交时间里对同一位置连写 5 和 6，快照查出来必须是 **6**——这条专测"同版本多次 set 只留最终值"的合并逻辑；同时 `get(0,0)` 还得是 0，因为快照 0 拍在两次写入之前。
  - `array.set(0,8); assert array.get(0,sid)==6`：拍完快照再改，旧快照的值纹丝不动——这条专测"不污染旧版本"，是可持久化的灵魂。
- **第二段（900 轮随机对拍）**：拿一个朴素的 `values` 列表当"当前状态"，每次 `snap` 时 `values[:]` 整份拷贝存进 `copies`（这就是那个"代价很大"的暴力参照物）。900 轮里随机做三件事：`set`（同时更新朴素列表）、`snap`（断言编号等于 `len(copies)`，验证编号从 0 连续递增；然后拷贝存档）、`get`（随机挑一个位置和一个已有版本，断言我们的快照数组与暴力拷贝值相等）。注意 `elif copies:` 这个守卫——`get` 分支只在已经至少拍过一张快照时才执行，否则随机挑版本无从挑起，这是随机测试必须自己守住的边界。循环结束后还有一轮**全量核对**：对每个版本、每个位置逐一比对，随机抽查可能有漏网（900 轮里某对"版本×位置"可能从没被问到过），全量扫描保证无死角。暴力拷贝必然正确，所以它能当裁判。
- **第三段（结构不变量）**：`assert all(all(a[0]<b[0] for a,b in zip(history,history[1:])) ...)` 检查每本账本里版本号严格递增——这是二分正确性的前提，专抓"该合并的没合并，账本里出现同版本两条记录"这类隐患。读法提示：外层 `all(...)` 遍历所有位置的账本，内层 `zip(history, history[1:])` 把每本账本切成"相邻对"，`a[0]<b[0]` 要求每对前一条的版本号严格小于后一条。前两段测的是"输出对不对"，这一段测的是"内部状态长什么样"——输出全对但账本里攒了垃圾（重复版本记录），将来查询就可能翻车，所以结构检查和功能检查都要做。

**三个失效条件自查**（对着 “掌握检查”部分 的自测要求）：下标越界会抛 `IndexError`；查询未发放的 snap_id 会抛 `ValueError`；负长度构造直接抛 `ValueError`。三个都是主动报错，没有一个静默吞掉。

In [4]:
array = SnapshotArray(3)

assert array.snap() == 0 and array.get(1, 0) == 0

array.set(0, 5)

array.set(0, 6)

sid = array.snap()

assert array.get(0, sid) == 6 and array.get(0, 0) == 0

array.set(0, 8)

assert array.get(0, sid) == 6

from random import Random

rng = Random(150)

fast = SnapshotArray(7)

values = [0] * 7

copies = []

for _ in range(900):
    op = rng.randrange(3)
    if op == 0:
        i = rng.randrange(7)
        value = rng.randrange(-100, 101)
        fast.set(i, value)
        values[i] = value
    elif op == 1:
        assert fast.snap() == len(copies)
        copies.append(values[:])
    elif copies:
        i = rng.randrange(7)
        version = rng.randrange(len(copies))
        assert fast.get(i, version) == copies[version][i]

for version, values in enumerate(copies):
    for i, x in enumerate(values):
        assert fast.get(i, version) == x

assert all((all((a[0] < b[0] for a, b in zip(history, history[1:]))) for history in fast.history))

print('N150: 所有本章断言通过')

N150: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 合并同一snap前对同一索引的多次set。

**练习 2：** 比较完整拷贝与路径复制。

<details>
<summary>展开思路提示（不是完整答案）</summary>

- **练习 1（合并同一 snap 前对同一索引的多次 set）**：提示——本章实现其实已经做了合并（`set` 里 `history[-1][0]==self.current` 的原地覆盖分支），练习是让你说清它为什么对、并设计实验展示收益。手算示例可以直接用 “运行示例”部分 的 `set(0,6); set(0,7)`：账本从 `[(0,5),(1,6)]` 变 `[(0,5),(1,7)]` 而不是变 `[(0,5),(1,6),(1,7)]`。验证思路：故意写一个"每次 set 都无脑 append"的错误版本，对拍二者的 `get` 结果和账本长度差异。边界别忘了"两次 snap 之间一次都没写"和"同一位置在当前版本写完又写回旧值"这两种。
- **练习 2（比较完整拷贝与路径复制）**：提示——完整拷贝就是测试里 `copies.append(values[:])` 那个暴力参照物：每次快照 O(长度)，查询 O(1)，但空间是 O(长度 × 快照数)。拿具体数字算账：长度 1000、快照 1000 次，暴力要存 100 万个数，而稀疏账本若总共只改过 2000 次，只需 2000 条记录加 1000 条初始。路径复制是介于两者之间的通用技术：每次修改只复制"根到叶的一条路径"，子树共享——可以画一棵小线段树（比如 7 个叶子），改一个叶子，看新旧版本各自能共享哪些节点。边界考虑：连续修改同一个位置时路径复制和稀疏记录谁更省。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [5]:
# 完整实现：本单元格不依赖其他单元格。
from bisect import bisect_right

class SnapshotArray:

    def __init__(self, length):
        if length < 0:
            raise ValueError('nonnegative length required')
        self.history = [[(0, 0)] for _ in range(length)]
        self.current = 0

    def _check_index(self, index):
        if not 0 <= index < len(self.history):
            raise IndexError('index outside snapshot array')

    def set(self, index, val):
        self._check_index(index)
        history = self.history[index]
        if history[-1][0] == self.current:
            history[-1] = (self.current, val)
        else:
            history.append((self.current, val))

    def snap(self):
        result = self.current
        self.current += 1
        return result

    def get(self, index, snap_id):
        self._check_index(index)
        if not 0 <= snap_id < self.current:
            raise ValueError('snapshot must already be committed')
        history = self.history[index]
        position = bisect_right(history, snap_id, key=lambda item: item[0]) - 1
        return history[position][1]

# 示例与回归测试
array = SnapshotArray(3)

assert array.snap() == 0 and array.get(1, 0) == 0

array.set(0, 5)

array.set(0, 6)

sid = array.snap()

assert array.get(0, sid) == 6 and array.get(0, 0) == 0

array.set(0, 8)

assert array.get(0, sid) == 6

from random import Random

rng = Random(150)

fast = SnapshotArray(7)

values = [0] * 7

copies = []

for _ in range(900):
    op = rng.randrange(3)
    if op == 0:
        i = rng.randrange(7)
        value = rng.randrange(-100, 101)
        fast.set(i, value)
        values[i] = value
    elif op == 1:
        assert fast.snap() == len(copies)
        copies.append(values[:])
    elif copies:
        i = rng.randrange(7)
        version = rng.randrange(len(copies))
        assert fast.get(i, version) == copies[version][i]

for version, values in enumerate(copies):
    for i, x in enumerate(values):
        assert fast.get(i, version) == x

assert all((all((a[0] < b[0] for a, b in zip(history, history[1:]))) for history in fast.history))

print('N150: 所有本章断言通过')

N150: 所有本章断言通过


## 11. 代表题与迁移

- **1146. Snapshot Array**：就是本章 `SnapshotArray` 的原题——练"按位置稀疏记账 + snap 只发号 + 二分查时间前驱"这套设计，重点体会"快照不复制、修改才记录"的省法。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。